# Capítulo 19 · Laboratório 19: criptoativos
*Análise Técnica dos Mercados Financeiros*, 4ª edição, Flávio Lemos, CMT

Dados até **30/09/2026**. Na primeira execução, o notebook baixa os dados do Yahoo Finance e da Blockchain.com na sua máquina e guarda uma cópia local (`dados_cap19.pkl`), que as execuções seguintes reaproveitam. O Yahoo corrige o histórico (proventos, desdobramentos), por isso seus números podem diferir um pouco dos do livro. O laboratório mede os ciclos do halving, a volatilidade e a correlação do Bitcoin com o Nasdaq e o ouro, testa a média de 200 semanas, o múltiplo de Mayer e o TD Setup, compara os ETFs dos EUA e da B3 e o futuro da CME com a moeda, testa o "gap da CME", mede o fôlego de 20 moedas grandes (ideia de Jamie Coutts, CMT), usa dados da cadeia da Blockchain.com e mostra o que sobrou da febre de 2021.

In [ ]:
!pip install -q yfinance

In [ ]:
import json
import logging
import urllib.request
import os
import numpy as np
import pandas as pd
import yfinance as yf

logging.getLogger("yfinance").setLevel(logging.CRITICAL)
FIM = "2026-10-01"                      # dados até 30/09/2026
CACHE = "dados_cap19.pkl"
pc = lambda x, k=0: f"{x:.{k}%}".replace(".", ",")
ps = lambda x, k=0: ("+" if x >= 0 else "") + pc(x, k)
nr = lambda x, k=0: f"{x:,.{k}f}".replace(",", "X").replace(".", ",").replace("X", ".")
cr = lambda x: ("+" if x >= 0 else "") + f"{x:.2f}".replace(".", ",")
MOEDAS = ["ETH", "XRP", "BNB", "SOL", "DOGE", "ADA", "TRX", "LINK", "XLM", "BCH", "LTC", "AVAX", "DOT",
          "HBAR", "ATOM", "ETC", "XMR", "UNI", "AAVE", "FIL"]
FEBRE = ["DOGE", "MANA", "SAND", "AXS"]
TICKERS = ["BTC-USD", "^IXIC", "GLD", "^BVSP", "BRL=X", "IBIT", "FBTC", "ETHA", "QBTC11.SA", "BITH11.SA",
           "ETHE11.SA", "BTC=F"] + [f"{m}-USD" for m in MOEDAS + FEBRE[1:]]
CADEIA = {"hash": "hash-rate", "taxas_btc": "transaction-fees", "taxas_usd": "transaction-fees-usd",
          "transacoes": "n-transactions"}


def blockchain_com(nome):
    url = f"https://api.blockchain.info/charts/{nome}?timespan=all&format=json&sampled=false"
    for tentativa in range(5):                                   # o servidor às vezes demora ou derruba a conexão
        try:
            v = json.loads(urllib.request.urlopen(url, timeout=60).read())["values"]
            break
        except Exception:
            if tentativa == 4:
                raise RuntimeError("Não consegui baixar os dados da cadeia (Blockchain.com) depois de 5 tentativas. O site"
                                   " pode estar fora do ar. Espere alguns minutos e rode a célula de novo.")
    s_ = pd.Series({pd.to_datetime(x["x"], unit="s"): x["y"] for x in v})
    return s_.loc[:FIM]


def confere(baixados):
    """O Yahoo devolve uma tabela vazia quando falha: para aqui, com uma mensagem clara, sem gravar a cópia local."""
    for nome, x in baixados.items():
        if len(x) == 0:
            raise RuntimeError(f"Não consegui baixar {nome} do Yahoo Finance. O provedor pode estar fora do ar ou"
                               " limitando pedidos. Espere alguns minutos e rode a célula de novo.")


if os.path.exists(CACHE):
    D = pd.read_pickle(CACHE)
else:
    d = yf.download(TICKERS, start="2014-01-01", end=FIM, auto_adjust=True, progress=False)
    confere({"as criptomoedas, os ETFs e os índices": d})
    D = {k: d[k] for k in ("Open", "High", "Low", "Close")}
    D["cadeia"] = pd.DataFrame({k: blockchain_com(v) for k, v in CADEIA.items()})
    pd.to_pickle(D, CACHE)

## Parte 1: dados

In [ ]:
C = D["Close"]
btc = C["BTC-USD"].dropna()
btc = btc[btc > 0]
print(f"Parte 1. Bitcoin de {btc.index[0]:%d/%m/%Y} a {btc.index[-1]:%d/%m/%Y}: {nr(len(btc))} dias, todos os dias da"
      f" semana; último fechamento US$ {nr(btc.iloc[-1])}, {ps(btc.iloc[-1] / btc.max() - 1)} da máxima de"
      f" US$ {nr(btc.max())} ({btc.idxmax():%d/%m/%Y}).")

## Parte 2: ciclos do halving

In [ ]:
HALVINGS = ["2016-07-09", "2020-05-11", "2024-04-20"]     # o de 28/11/2012 é anterior aos dados do Yahoo
TAB_HALV = []
print("\nParte 2. Bitcoin depois de cada halving")
for h in map(pd.Timestamp, HALVINGS):
    x = btc.loc[h:h + pd.Timedelta(days=730)]
    topo = x.idxmax()
    depois = btc.loc[topo:topo + pd.Timedelta(days=500)]
    lin = (h, x.iloc[0], topo, (topo - h).days, x.max(), x.max() / x.iloc[0] - 1, depois.min() / x.max() - 1,
           depois.idxmin())
    TAB_HALV.append(lin)
    print(f"   halving {h:%d/%m/%Y}: US$ {nr(lin[1])} | topo em {topo:%d/%m/%Y} ({lin[3]} dias depois), US$ {nr(lin[4])}"
          f" ({ps(lin[5])}) | maior queda depois do topo: {pc(lin[6])} (fundo em {lin[7]:%d/%m/%Y})")

In [ ]:
import matplotlib.pyplot as plt
ax = btc.plot(figsize=(8, 3.5), logy=True, color="black", title="Bitcoin (escala log) e os halvings")
for h in HALVINGS:
    ax.axvline(pd.Timestamp(h), color="gray", ls="--")
plt.show()

## Parte 3: maturidade

In [ ]:
DIAS = pd.DataFrame({"btc": btc, "nasdaq": C["^IXIC"], "ouro": C["GLD"], "ibov": C["^BVSP"]}).ffill(limit=3)
UTEIS = DIAS[DIAS.index.dayofweek < 5].pct_change(fill_method=None)   # só dias úteis, para comparar com a bolsa
TAB_MAT = []
print("\nParte 3. Volatilidade anualizada (dias úteis) e correlação diária do Bitcoin, por ano")
print("   ano    BTC  Nasdaq  ouro  Ibovespa | corr. Nasdaq  corr. ouro")
for ano in range(2015, 2027):
    x = UTEIS.loc[str(ano)].dropna()
    v = x.std() * np.sqrt(252)
    lin = (ano, v["btc"], v["nasdaq"], v["ouro"], v["ibov"], x["btc"].corr(x["nasdaq"]), x["btc"].corr(x["ouro"]))
    TAB_MAT.append(lin)
    print(f"   {ano}  {pc(lin[1]):>5} {pc(lin[2]):>6} {pc(lin[3]):>5} {pc(lin[4]):>7}   |    {cr(lin[5]):>6}"
          f"       {cr(lin[6]):>6}")

## Parte 4: múltiplo de Mayer e média de 200 semanas

In [ ]:
MAYER = btc / btc.rolling(200).mean()
F365 = btc.shift(-365) / btc - 1
TAB4 = pd.DataFrame({"m": MAYER, "f": F365}).dropna()
TAB_MAYER = []
print(f"\nParte 4. Múltiplo de Mayer hoje: {nr(MAYER.iloc[-1], 2)}. Bitcoin um ano depois, por faixa (desde 2015):")
for nome, cond in [("abaixo de 0,8", TAB4["m"] < 0.8), ("0,8 a 1,5", TAB4["m"].between(0.8, 1.5)),
                   ("1,5 a 2,4", TAB4["m"].between(1.5, 2.4, inclusive="right")), ("acima de 2,4", TAB4["m"] > 2.4)]:
    x = TAB4.loc[cond, "f"]
    TAB_MAYER.append((nome, len(x), x.median(), (x > 0).mean()))
    print(f"   {nome:<14} {nr(len(x)):>5} dias | mediana {ps(x.median())} | positivo em {pc((x > 0).mean())}")
SEM = btc.resample("W-SUN").last()
MM200S = SEM.rolling(200).mean()
ABAIXO = (SEM < MM200S)[MM200S.notna()]
print(f"   Média de 200 semanas hoje: US$ {nr(MM200S.iloc[-1])}; fechamentos semanais abaixo dela desde"
      f" {MM200S.dropna().index[0]:%Y}: {pc(ABAIXO.mean())} das semanas ({ABAIXO.sum()} semanas)")

## Parte 5: TD Setup no Bitcoin

In [ ]:
compra = (btc < btc.shift(4)).astype(int)
venda = (btc > btc.shift(4)).astype(int)
CONT_C = compra.groupby((compra == 0).cumsum()).cumsum()
CONT_V = venda.groupby((venda == 0).cumsum()).cumsum()
TAB_TD = []
print("\nParte 5. TD Setup (9) no Bitcoin diário: retorno médio depois do sinal (entre parênteses, % de altas)")
for nome, cond in [("9 de compra", CONT_C == 9), ("9 de venda", CONT_V == 9), ("qualquer dia", btc.notna())]:
    lin = [nome, int(cond.sum())]
    for h in (5, 10, 20):
        x = (btc.shift(-h) / btc - 1)[cond].dropna()
        lin += [x.mean(), (x > 0).mean()]
    TAB_TD.append(lin)
    print(f"   {nome:<13} {lin[1]:>5} sinais | 5d {ps(lin[2], 1)} ({pc(lin[3])}) | 10d {ps(lin[4], 1)} ({pc(lin[5])})"
          f" | 20d {ps(lin[6], 1)} ({pc(lin[7])})")

## Parte 6: ETFs e futuros regulados

In [ ]:
BRL = C["BRL=X"].dropna()
BTC_BRL = (btc * BRL.reindex(btc.index).ffill()).dropna()
ETH = C["ETH-USD"].dropna()
ETH_BRL = (ETH * BRL.reindex(ETH.index).ffill()).dropna()
TAB_ETF = []
print("\nParte 6. Produtos regulados contra a moeda (retornos diários nos dias em que os dois têm preço)")
for etf, ref, nome in [("IBIT", btc, "IBIT (EUA) x Bitcoin em dólar"), ("FBTC", btc, "FBTC (EUA) x Bitcoin em dólar"),
                       ("ETHA", ETH, "ETHA (EUA) x Ether em dólar"), ("BTC=F", btc, "Futuro CME x Bitcoin"),
                       ("QBTC11.SA", BTC_BRL, "QBTC11 (B3) x Bitcoin em reais"),
                       ("BITH11.SA", BTC_BRL, "BITH11 (B3) x Bitcoin em reais"),
                       ("ETHE11.SA", ETH_BRL, "ETHE11 (B3) x Ether em reais")]:
    dd = pd.DataFrame({"e": C[etf], "b": ref}).dropna()
    r = dd.pct_change().dropna()
    te, tb = dd["e"].iloc[-1] / dd["e"].iloc[0] - 1, dd["b"].iloc[-1] / dd["b"].iloc[0] - 1
    TAB_ETF.append((nome, dd.index[0], r["e"].corr(r["b"]), te, tb))
    print(f"   {nome:<32} desde {dd.index[0]:%d/%m/%Y}: correlação {nr(r['e'].corr(r['b']), 2)} | produto {ps(te)}"
          f" x moeda {ps(tb)} | diferença {ps(te - tb, 1)}")

## Parte 7: o gap da CME

In [ ]:
FUT = pd.DataFrame({k: D[k]["BTC=F"] for k in ("Open", "High", "Low", "Close")}).dropna()
FUT = FUT[FUT.index < "2026-05-29"]                     # a partir de 29/05/2026 a CME negocia cripto 24 horas por dia
seg = FUT.index.to_series().diff().dt.days >= 3        # primeira sessão depois do fim de semana (ou feriado)
GAPS = []
for i in np.where(seg.values)[0]:
    ant, hoje = FUT.iloc[i - 1], FUT.iloc[i]
    nivel = ant["Close"]
    if hoje["Low"] > nivel:                              # gap de alta: abriu acima e não voltou no primeiro dia
        tipo = "alta"
    elif hoje["High"] < nivel:
        tipo = "baixa"
    else:
        continue                                        # sem gap ou fechado no mesmo dia
    prox = FUT.iloc[i + 1:i + 21]
    toca = (prox["Low"] <= nivel) if tipo == "alta" else (prox["High"] >= nivel)
    dias = int(np.argmax(toca.values)) + 1 if toca.any() else np.nan
    GAPS.append((FUT.index[i], tipo, abs(hoje["Open"] / nivel - 1), dias))
G = pd.DataFrame(GAPS, columns=["data", "tipo", "tam", "dias"])
TAB_GAP = []
print(f"\nParte 7. Gap da CME: {len(G)} gaps de fim de semana não fechados no primeiro pregão"
      f" ({FUT.index[0]:%m/%Y} a {FUT.index[-1]:%m/%Y}; de {seg.sum()} recomeços de semana)")
for nome, x in [("todos", G), ("gaps de alta", G[G["tipo"] == "alta"]), ("gaps de baixa", G[G["tipo"] == "baixa"])]:
    lin = (nome, len(x), x["tam"].median(), (x["dias"] <= 5).mean(), (x["dias"] <= 20).mean())
    TAB_GAP.append(lin)
    print(f"   {nome:<14} {lin[1]:>4} | tamanho mediano {pc(lin[2], 1)} | fechados em 5 pregões: {pc(lin[3])}"
          f" | em 20 pregões: {pc(lin[4])}")

## Parte 8: fôlego cripto

In [ ]:
ALT = C[[f"{m}-USD" for m in MOEDAS]]
ALT = ALT.where(ALT > 0)
MM = ALT.rolling(200, min_periods=200).mean()
ACIMA = (ALT > MM).where(MM.notna() & ALT.notna())
PCT = ACIMA.mean(axis=1)[ACIMA.notna().sum(axis=1) >= 15]          # só dias com pelo menos 15 moedas com dado
VAR90 = ALT.div(btc, axis=0).pct_change(90, fill_method=None)
MELHOR = (VAR90 > 0).where(VAR90.notna()).mean(axis=1)[VAR90.notna().sum(axis=1) >= 15]
TAB_FOL = []
print(f"\nParte 8. Fôlego de 20 moedas grandes (só as que sobreviveram até hoje: viés de sobrevivência)")
print(f"   Hoje ({PCT.index[-1]:%d/%m/%Y}): {pc(PCT.iloc[-1])} acima da média de 200 dias |"
      f" {pc(MELHOR.iloc[-1])} melhores que o Bitcoin em 90 dias")
for ano in range(max(PCT.index[0].year, MELHOR.index[0].year), 2027):
    a, b = PCT.loc[str(ano)].mean(), MELHOR.loc[str(ano)].mean()
    TAB_FOL.append((ano, a, b))
    print(f"   {ano}: média de {pc(a)} acima da MM200 | {pc(b)} batendo o Bitcoin em 90 dias")

In [ ]:
import matplotlib.pyplot as plt
PCT.rolling(10).mean().plot(figsize=(8, 3), color="black", title="Moedas grandes acima da média de 200 dias")
plt.axhline(0.5, color="gray", ls="--")
plt.show()

## Parte 9: dados na cadeia

In [ ]:
CAD = D["cadeia"].loc["2015":].resample("ME").mean()
CAD["preco"] = btc.resample("ME").mean()
VAR = np.log(CAD).diff().dropna()
TAB_CAD = {k: VAR["preco"].corr(VAR[k]) for k in ("taxas_usd", "taxas_btc", "transacoes", "hash")}
HASH = D["cadeia"]["hash"].dropna()
print(f"\nParte 9. Dados na cadeia (médias mensais desde 2015; correlação das variações mensais com o preço do Bitcoin)")
for k, nome in [("taxas_usd", "taxas pagas, em dólar"), ("taxas_btc", "taxas pagas, em bitcoins"),
                ("transacoes", "número de transações"), ("hash", "taxa de hash")]:
    print(f"   {nome:<28} {cr(TAB_CAD[k])}")
print(f"   Taxa de hash: {nr(HASH.loc[:'2021-12-31'].max() / 1e6)} milhões de TH/s no pico de 2021 e"
      f" {nr(HASH.rolling(30).mean().iloc[-1] / 1e6)} milhões de TH/s hoje (média de 30 dias),"
      f" {nr(HASH.rolling(30).mean().iloc[-1] / HASH.loc[:'2021-12-31'].max(), 1)} vezes mais")

## Parte 10: a febre de 2021

In [ ]:
TAB_FEBRE = []
print("\nParte 10. A febre de 2021: do pico até 30/09/2026")
for m in ["BTC", "ETH"] + FEBRE:
    s_ = C[f"{m}-USD"].dropna()
    s_ = s_[s_ > 0]
    pico = s_.loc["2021"].idxmax()
    TAB_FEBRE.append((m, pico, s_[pico], s_.iloc[-1], s_.iloc[-1] / s_[pico] - 1))
    print(f"   {m:<5} pico de 2021 em {pico:%d/%m/%Y} (US$ {nr(s_[pico], 4 if s_[pico] < 1 else 2)}) | hoje"
          f" US$ {nr(s_.iloc[-1], 4 if s_.iloc[-1] < 1 else 2)} | {ps(s_.iloc[-1] / s_[pico] - 1)}")

## Peça à IA

O Colab tem um assistente de IA. Abra o assistente, cole um destes pedidos e confira a resposta rodando a célula indicada.

1. **Explicar:** "Explique, linha por linha, a célula da Parte 4. Mostre por que `btc.rolling(200)` conta 200 dias corridos, por que `shift(-365)` só mede o resultado um ano depois e não entra no sinal, e por que a média de 200 semanas usa `resample('W-SUN')`." Confira com a Parte 4, que imprime o múltiplo de Mayer de hoje (1,17 no livro), as quatro faixas da Tabela 19.2 e a média de 200 semanas (US$ 66.166).
2. **Adaptar:** "Numa célula nova, depois da Parte 4, troque `btc` por uma moeda da lista `MOEDAS`, com `C['[MOEDA]-USD'].dropna()` e só os preços maiores que zero, e repita as contas de `MAYER`, `F365` e das quatro faixas. Não mude a média de 200 dias, o horizonte de 365 dias nem os limites 0,8, 1,5 e 2,4." Antes de olhar a moeda nova, confira se a Parte 4 do Bitcoin continua com os números do livro (abaixo de 0,8: 547 dias, mediana de +50%, positivo em 78%). As faixas foram medidas no Bitcoin.

Mais pedidos deste capítulo (entender e aplicar ao seu gráfico) estão na página do capítulo, no hub do livro.

## Para ir além
1. Na Parte 4, troque o horizonte de 365 para 90 dias. O múltiplo de Mayer acima de 2,4 continua avisando queda?
2. Na Parte 5, teste o 9 de compra só quando o preço está acima da média de 200 dias.
3. Na Parte 7, conte só os gaps maiores que 3%. Eles fecham mais ou menos vezes?
4. Na Parte 8, troque as 20 moedas pelas 10 maiores de hoje. O fôlego muda?

**Governança:** os resultados deste notebook valem só para os dados e as datas usadas. Mudar regras até o resultado ficar bonito é sobreajuste (*overfitting*): a regra decora o passado e falha no futuro.

━━━━━━━━━━━━━━━━━━━━━━━

⚠️ AVISO LEGAL (DISCLAIMER):

━━━━━━━━━━━━━━━━━━━━━━━

Todo o conteúdo deste livro  é estritamente educacional e informativo. 

Não constitui, em nenhuma hipótese, recomendação de compra ou venda de ativos financeiros. 

Testes passados não são garantias de resultados futuros.

Investimentos envolvem riscos, incluindo a perda total do capital investido. 

Consulte sempre um profissional habilitado antes de tomar qualquer decisão de investimento.